# Week 74, Day 1: Style and PEP 8

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w074d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def ship_a(order):
    if order:
        if order["paid"]:
            if order["items"]:
                if not order["blocked"]:
                    return "ship"
                else:
                    return "blocked"
            else:
                return "empty"
        else:
            return "unpaid"
    return "none"

def ship_b(order):
    if not order:
        return "none"
    if not order["paid"]:
        return "unpaid"
    if not order["items"]:
        return "empty"
    if order["blocked"]:
        return "blocked"
    return "ship"

orders = [None, {"paid": False}, {"paid": True, "items": []}, {"paid": True, "items": [1], "blocked": True}, {"paid": True, "items": [1], "blocked": False}]
print([ship_a(o) for o in orders] == [ship_b(o) for o in orders], [ship_b(o) for o in orders])

In [ ]:
import ast

DECISIONS = (ast.If, ast.For, ast.While, ast.IfExp, ast.ExceptHandler, ast.AsyncFor)

def complexity(func):
    score = 1
    for node in ast.walk(func):
        if isinstance(node, DECISIONS):
            score += 1
        elif isinstance(node, ast.BoolOp):
            score += len(node.values) - 1
        elif isinstance(node, ast.comprehension):
            score += len(node.ifs)
    return score

source = '''
def grade(score, bonus):
    if score < 0 or score > 100:
        raise ValueError(score)
    total = score + (bonus if bonus > 0 else 0)
    for limit, name in [(70, "distinction"), (40, "pass")]:
        if total >= limit:
            return name
    return "fail"
'''
for node in ast.parse(source).body:
    print(node.name, complexity(node))

In [ ]:
import json

# Look before you leap: two checks, and still not safe (the file could change in between).
# EAFP: try the operation and handle exactly the failures you expect.
def load_settings(text):
    try:
        settings = json.loads(text)
    except json.JSONDecodeError:
        return {}
    return settings if isinstance(settings, dict) else {}

names = ["ada", "bob", "cy"]
scores = [91, 78, 85]
print([f"{i}. {name.title()}: {score}" for i, (name, score) in enumerate(zip(names, scores, strict=True), start=1)])
print(load_settings('{"theme": "dark"}'), load_settings("not json"))

In [ ]:
import io, tokenize

source = 'if x == None:  # check x == None\n    print("x == None")\n'

naive = source.replace("== None", "is None")
print(naive)

def fix_none(text):
    tokens = list(tokenize.generate_tokens(io.StringIO(text).readline))
    lines = text.splitlines(keepends=True)
    for i, tok in enumerate(tokens[:-1]):
        nxt = tokens[i + 1]
        if tok.type == tokenize.OP and tok.string in ("==", "!=") and nxt.type == tokenize.NAME and nxt.string == "None":
            row, col = tok.start
            new = "is" if tok.string == "==" else "is not"
            lines[row - 1] = lines[row - 1][:col] + new + lines[row - 1][col + 2:]
    return "".join(lines)

print(fix_none(source))

> **Key idea.** Readable code has names at the level of the problem, small functions that do one thing, shallow nesting (guard clauses), named constants and comments that say why. Cyclomatic complexity is 1 plus each if, loop, except, conditional expression, extra and/or operand, comprehension if and case; with nesting depth and size it signals code to look at, not a target. Use enumerate, zip, comprehensions, truthiness, unpacking, dict.get, EAFP, with and f-strings when they make intent clearer. Configure linters in pyproject.toml with selected rule groups, per-file ignores and named noqa comments. Automatic fixes must change only what they mean to, so work on tokens or the tree, never raw text.

### Exercise: Cyclomatic complexity

Write `complexities(source)`: a dict from each function's name (including methods, by their plain name, and nested functions) to its cyclomatic complexity: 1, plus one for each `If` (which includes `elif`), `For`, `AsyncFor`, `While`, `IfExp`, `ExceptHandler` and `match_case`, plus `len(values) - 1` for each `BoolOp`, plus the number of `ifs` in each comprehension. Count only the function's **own** body: decisions inside a nested function count for the nested one, not its parent. Then `too_complex(source, limit=10)`: the sorted names whose complexity is over `limit`.

In [ ]:
import ast

def complexities(source):
    ___

def too_complex(source, limit=10):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_mccabe():\n    \"counting decision points\"\n    src = '''\ndef simple(x):\n    return x + 1\n\ndef branchy(x, y):\n    if x and y or x:\n        return 1\n    elif x > 3:\n        return [i for i in range(x) if i % 2 if i > 1]\n    while y:\n        y -= 1\n    try:\n        pass\n    except ValueError:\n        pass\n    except KeyError:\n        pass\n    return 2 if x else 3\n\ndef outer(n):\n    def inner(m):\n        if m:\n            return m\n        return 0\n    for i in range(n):\n        pass\n    return inner\n\nclass Shape:\n    def area(self, kind):\n        match kind:\n            case \"square\":\n                return 1\n            case \"circle\":\n                return 3\n            case _:\n                return 0\n'''\n    assert complexities(src) == {\"simple\": 1, \"branchy\": 11, \"outer\": 2, \"inner\": 2, \"area\": 4}\n    assert too_complex(src) == [\"branchy\"] and too_complex(src, limit=1) == [\"area\", \"branchy\", \"inner\", \"outer\"]")

### Exercise: How deep does it go?

Write `max_depth(source)`: a dict from each top-level function's name to the deepest nesting of **blocks** inside it, where a block is the body of an `if`, `for`, `while`, `with`, `try` (its body, handlers, `else` and `finally` all count as one level) or `match` case. A function with no blocks has depth 0; an `elif` is at the **same** depth as its `if`.

In [ ]:
import ast

def max_depth(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_depth():\n    \"nesting levels\"\n    src = '''\ndef flat(x):\n    return x\n\ndef one(x):\n    if x:\n        return 1\n    elif x is None:\n        return 2\n    else:\n        return 3\n\ndef deep(rows):\n    for row in rows:\n        if row:\n            with open(row) as f:\n                try:\n                    while True:\n                        pass\n                except OSError:\n                    pass\n    return None\n'''\n    assert max_depth(src) == {\"flat\": 0, \"one\": 1, \"deep\": 5}")

### Exercise: Unexplained numbers

Write `magic_numbers(source)`: a sorted list of `(line, value)` for every int or float constant used **inside a function** other than `0`, `1`, `-1` (a unary minus on 1), `2` and `100`, except values in a default argument, and except when the constant is the whole value assigned to an `UPPER_CASE` name. (Booleans don't count.) Module-level constants are fine wherever they are.

In [ ]:
import ast

ALLOWED = {0, 1, -1, 2, 100}

def magic_numbers(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_magic():\n    \"numbers that deserve names\"\n    src = '''PASS_MARK = 0.73\nLIMIT = 500\n\ndef grade(score, scale=10):\n    BONUS = 5\n    if score > 0.73 * 100:\n        return score * 2 + 17\n    if score < -1 or score > LIMIT:\n        return -3\n    return round(score / 3.5, 1) + True\n'''\n    assert magic_numbers(src) == [(6, 0.73), (7, 17), (9, -3), (10, 3.5)]")

### Exercise: Saying it the Python way

Rewrite the three functions in the starter so they behave exactly the same but read the Python way: no `range(len(...))`, no index variables used only to look items up, no `== True`, `== None` or `len(...) == 0`. The checks run them and read your code.

In [ ]:
def numbered(items):
    out = []
    for i in range(len(items)):
        out.append(str(i + 1) + ". " + items[i])
    return out

def dot(a, b):
    total = 0
    for i in range(len(a)):
        total = total + a[i] * b[i]
    return total

def first_missing(record, keys):
    if len(keys) == 0:
        return None
    for key in keys:
        if (key in record) == True:
            continue
        if record.get(key) == None:
            return key
    return None

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pythonic():\n    \"same behaviour, clearer code\"\n    assert numbered([\"tea\", \"jam\"]) == [\"1. tea\", \"2. jam\"] and numbered([]) == []\n    assert dot([1, 2, 3], [4, 5, 6]) == 32 and dot([], []) == 0\n    assert first_missing({\"a\": 1, \"b\": None}, [\"a\", \"c\", \"b\"]) == \"c\" and first_missing({}, []) is None\n    assert first_missing({\"a\": 1}, [\"a\"]) is None\n    for bad in (\"range(len(\", \"== True\", \"== None\", \"len(keys) == 0\"):\n        assert bad not in source, f\"still uses {bad}\"\n    assert \"enumerate\" in source and \"zip\" in source")

### Exercise: Rules, ignores and noqa

A linter has produced `findings`: a list of `(path, line, code)`. Write `apply_config(findings, sources, select, ignore, per_file)`, keeping a finding only if: its code starts with one of the prefixes in `select`; it doesn't start with one in `ignore`; it isn't ignored for its file by `per_file` (a dict of glob pattern to a list of prefixes, matched with `fnmatch`); and its line doesn't end with a `# noqa` comment naming it. A noqa comment looks like `# noqa: E711, F401` (codes separated by commas, matched by exact code); a bare `# noqa` silences nothing (return those lines' findings, as real tools would if configured strictly). `sources` maps paths to file text. Return the kept findings, sorted.

In [ ]:
import fnmatch, re

def apply_config(findings, sources, select, ignore, per_file):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_config():\n    \"selecting and silencing\"\n    sources = {\n        \"src/app.py\": \"import os  # noqa: F401\\nx = None\\nif x == None:  # noqa\\n    pass\\nif x == None:  # noqa: E711\\n    pass\\n\",\n        \"tests/test_app.py\": \"assert True\\nimport sys\\n\",\n    }\n    findings = [(\"src/app.py\", 1, \"F401\"), (\"src/app.py\", 3, \"E711\"), (\"src/app.py\", 5, \"E711\"), (\"src/app.py\", 2, \"N806\"),\n                (\"tests/test_app.py\", 1, \"S101\"), (\"tests/test_app.py\", 2, \"F401\"), (\"src/app.py\", 4, \"E501\")]\n    kept = apply_config(findings, sources, select=[\"E\", \"F\", \"S\"], ignore=[\"E501\"], per_file={\"tests/*\": [\"S101\"]})\n    assert kept == [(\"src/app.py\", 3, \"E711\"), (\"tests/test_app.py\", 2, \"F401\")]")

### Bug hunt: A guard clause the wrong way round

Flattening `discount` with guard clauses, someone inverted the member check, so members now get **no** discount and everyone else gets one: this prints `[100, 90, 40]`. Fix the guard so it prints `[90, 100, 40]` (a member, not a member, and a member whose order is too small).

In [ ]:
def discount(price, member):
    if member:
        return price
    if price < 50:
        return price
    return price * 0.9

print([round(discount(100, True)), round(discount(100, False)), round(discount(40, True))])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the same behaviour as before\"\n    expect_output(\"[90, 100, 40]\")")

### Bug hunt: A fix that changed a string

The fixer replaces the text `== None` everywhere, so it also rewrites the message inside the string, and the program prints the changed source. Use `tokenize` so only the operator changes, and this prints `if value is None: print("value == None")`.

In [ ]:
import io, tokenize

source = 'if value == None: print("value == None")'
fixed = source.replace("== None", "is None")
print(fixed)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"change code, not strings\"\n    expect_output('if value is None: print(\"value == None\")')\n    assert \"tokenize.\" in source.split(\"source =\")[1]")

### Bug hunt: A complexity that forgot and and or

The counter adds one for each `if` but ignores boolean operators, so `x and y or z` adds nothing and this prints `2`. Count each extra operand of every `BoolOp` (an `and` or `or` with three operands adds two), so this prints `4`.

In [ ]:
import ast

def complexity(source):
    score = 1
    for node in ast.walk(ast.parse(source)):
        if isinstance(node, ast.If):
            score += 1
    return score

print(complexity("if x and y or z:\n    pass\n"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"boolean operators are decisions\"\n    expect_output(\"4\")")

### Puzzle: Guard clauses

Arrange the lines of a function that checks its inputs with guard clauses and leaves the main work for the end. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the cyclomatic complexity of: def f(x): return 1 if x else 2
q = ___   # the complexity of a function with one for loop containing one if, and nothing else
r = ___   # True or False: a bare "# noqa" is a good way to silence one rule
s = ___   # the idiom for "for i in range(len(xs)): use i and xs[i]"
t = ___   # the acronym for "try it and handle the failure" (as opposed to "look before you leap")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 2 and q == 3 and r is False, \"p, q and r\"\n    assert s == \"enumerate\" and t == \"EAFP\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a code health report

Write `health(source, max_complexity=10, max_depth=3, max_lines=30, max_args=5)`: for every function and method (including nested ones), measure its **complexity** (as in the first exercise, counting only its own body), its **depth** (as in the second, applied to its own body, ignoring nested functions' insides), its **lines** (`end_lineno - lineno + 1`) and its **args** (all parameters, positional, keyword-only, `*args` and `**kwargs`, not counting `self` or `cls` when it's the first parameter of a method in a class). Return a list of report lines, one per function with at least one problem, sorted by the number of problems (most first), then by name: `"NAME: complexity 12 > 10, depth 4 > 3"`, listing only the measures over their limits, in the order complexity, depth, lines, args.

In [ ]:
import ast

def health(source, max_complexity=10, max_depth=3, max_lines=30, max_args=5):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_health():\n    \"several measures together\"\n    src = '''\nclass Shop:\n    def checkout(self, cart, user, coupon, address, payment, gift=False):\n        for item in cart:\n            if item:\n                for part in item:\n                    if part:\n                        pass\n        return 1\n\ndef fine(x):\n    return x\n\ndef tangled(a, b, c):\n    if a and b and c:\n        return 1\n    if a or b:\n        return 2\n    if a:\n        return 3\n    while b:\n        b -= 1\n    for x in c:\n        if x:\n            pass\n    return 4 if a else 5\n'''\n    report = health(src, max_complexity=8)\n    assert report == [\"checkout: depth 4 > 3, args 6 > 5\", \"tangled: complexity 11 > 8\"], report\n    assert health(src, max_complexity=20, max_depth=10, max_lines=100, max_args=10) == []\n    long_src = \"def long():\\n\" + \"    x = 1\\n\" * 40\n    assert health(long_src) == [\"long: lines 41 > 30\"]")

## Warm-up

The two functions do exactly the same thing. The second has no nesting at all: each **guard clause** deals with one case and gets out of the way, so the happy path is the last line instead of the deepest. No formatter would turn the first into the second; that's a decision about **structure**, and it's what today is about.

## Readable beyond the layout

Code is read far more often than it's written, by people (including you, later) trying to change it safely. The things that matter most aren't spacing:

- **Names that say what things are**, at the level of the problem: `overdue_invoices`, not `lst2`; `retry_delay_seconds`, not `d`. A function's name says what it does (`send_reminder`), a boolean's reads as a question (`is_overdue`, `has_access`). Short names are fine for short scopes: `for i, row in enumerate(rows)`.
- **Small functions that do one thing**, at one level of detail. A function you can't name without "and" probably does two things.
- **Shallow nesting.** Every level of indentation is one more condition to hold in your head. Guard clauses, early `continue`, and extracting a helper function flatten it.
- **No unexplained numbers.** `if age >= 18` is clear; `if score > 0.73` isn't. Give such a value a name (`PASS_MARK = 0.73`) so it's explained and changed in one place.
- **Comments that say why**, not what: `# the API counts from 1` helps; `# add one` doesn't. If code needs a comment to explain **what** it does, try renaming or restructuring it first.
- **The Zen of Python** (`import this`): "Readability counts", "Flat is better than nested", "Explicit is better than implicit", "Errors should never pass silently". They're principles for judgement, not rules for a checker.

## Measuring complexity

Some of this can be measured, which is how linters warn about it:

- **Cyclomatic complexity** (McCabe) counts the independent paths through a function: start at **1** and add one for each decision point: `if` and `elif`, `for` and `while` loops, `except` clauses, conditional expressions (`a if c else b`), each extra operand of `and` / `or`, each `if` in a comprehension, and each `case` of a `match`. A function at 1 has one path; above about 10, it's hard to test every path (each one needs a test case), and linters flag it (Ruff's rule C901).
- **Nesting depth**: the deepest level of blocks inside a function. Beyond 3 or 4 levels, readers lose track.
- **Size**: the number of lines and of parameters. Very long functions and long parameter lists both suggest a function doing too much.

The `ast` module (Week 22 Day 4, and Week 64) makes all three easy to compute. They're signals for a human to look at, not targets: a long `match` over twenty commands can be perfectly clear.

## Idioms that read well

Idiomatic Python ("Pythonic" code) uses the language's features so the intent shows:

| Instead of | Write |
|---|---|
| `for i in range(len(items)): print(i, items[i])` | `for i, item in enumerate(items):` |
| `for i in range(len(a)): total += a[i] * b[i]` | `for x, y in zip(a, b):` (`zip(..., strict=True)` when lengths must match) |
| a loop that appends to a list | a comprehension, when it's a simple map or filter |
| `if x == True:` / `if len(items) == 0:` | `if x:` / `if not items:` |
| `t = a; a = b; b = t` | `a, b = b, a` |
| `if key in d: v = d[key] else: v = default` | `v = d.get(key, default)` |
| checking first, then acting (look before you leap) | trying, and handling the failure (**EAFP**: it's easier to ask forgiveness than permission), where checking would be racy or repeat the work |
| opening and closing by hand | `with` (Week 61) |
| `"Hello " + name + "!"` | `f"Hello {name}!"` |

Don't overdo it. A comprehension with two loops and three conditions is less readable than the loop it replaced; `functools.reduce` is rarely clearer than a `for` loop. Pythonic means **clear to a Python reader**, not short.

## Configuring and fixing

Real projects configure their linter in `pyproject.toml` (Ruff shown; Flake8 and Pylint are similar):

```toml
[tool.ruff]
line-length = 88

[tool.ruff.lint]
select = ["E", "F", "B", "I", "UP", "C901", "N"]   # rule groups to enable
ignore = ["E501"]                                  # leave line length to the formatter

[tool.ruff.lint.per-file-ignores]
"tests/*" = ["S101"]                               # assert is fine in tests

[tool.ruff.lint.mccabe]
max-complexity = 10
```

- Enable rule **groups** deliberately, starting small on an old codebase and widening over time.
- **`# noqa: CODE`** at the end of a line silences one rule on that line, for a deliberate exception. Always name the code (a bare `# noqa` hides everything), and prefer fixing the code.
- **Per-file ignores** suit whole categories of file: tests, generated code, migration scripts.

Many rules come with **automatic fixes** (`ruff check --fix`): replacing `== None` with `is None`, sorting imports, removing unused imports, upgrading old syntax. A safe fixer changes the code it means to change and **nothing else**: not the comments, not the spacing, and certainly not the inside of a string that happens to contain the same characters. That's why fixers work on **tokens** (with `tokenize`) or the syntax tree, not on raw text.

The naive replacement changed the comment and the string too; the token-based fix changed only the operator. (Real fixers also re-check positions after each change; this one works because each line has at most one fix.)

## Practice

## Quiz

## Daily challenge

## Before you go

Run today's health report on the longest function you've written in this course, and flatten it with guard clauses or a helper. Tomorrow: **code review**: reading other people's changes, writing comments that help, and the habits that make a pull request easy to review.